In [1]:
from ast import literal_eval  # as tuple literal_eval -> convert string representation of list to actual list
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display


In [2]:
# --- par‑plant ---------------------------------------------------------------
path_box = Path("/home/loai/Documents/code/RSMLExtraction/Results/RPE1/results_per_plant.csv")
df_plant_wid = pd.read_csv(path_box)
df_plant_wid['root_ids'] = df_plant_wid['root_ids'].apply(literal_eval)
df_plant_wid

,model,split,box,metric,time,root_ids,source,value
0,Unet_bce_180,Val,230629PN031,NumberOfOrgans,1,"(86, 84, 81)",Prediction,1.000000
1,Unet_bce_180,Val,230629PN031,NumberOfOrgans,1,"(86, 84, 81)",expertized,1.000000
2,Unet_bce_180,Val,230629PN031,NumberOfOrgans,1,"(86, 84, 81)",before_expertized,1.000000
3,Unet_bce_180,Val,230629PN031,NumberOfOrgans,1,"(48, 47, 45)",Prediction,1.000000
4,Unet_bce_180,Val,230629PN031,NumberOfOrgans,1,"(48, 47, 45)",expertized,1.000000
...,...,...,...,...,...,...,...,...
603385,Unet_dice_120,Test,230629PN018,LateralRootLength,29,"(4, 7, 4)",expertized,102.080352
603386,Unet_dice_120,Test,230629PN018,LateralRootLength,29,"(4, 7, 4)",before_expertized,16.498474
603387,Unet_dice_120,Test,230629PN018,LateralRootLength,29,"(1, 1, 1)",Prediction,170.444195
603388,Unet_dice_120,Test,230629PN018,LateralRootLength,29,"(1, 1, 1)",expertized,317.155933


In [3]:
df_grouped = (
    df_plant_wid
    .set_index(['model', 'split', 'box', 'metric', 'root_ids', 'source', 'time'])
    .sort_index(level='time')
)
df_grouped

value
model         split box         metric           root_ids     source            time             
Unet_bce_063  Test  230629PN018 Convex_Area_Hull (12, 1, 1)   Prediction        1       59.458771
                                                              before_expertized 1      443.501066
                                                              expertized        1      443.501066
                                                 (14, 23, 13) Prediction        1      122.135427
                                                              before_expertized 1      371.522435
...                                                                                           ...
Unet_dice_260 Val   230629PN031 TotalRootLength  (34, 66, 63) before_expertized 29    2846.313710
                                                              expertized        29    2846.313710
                                                 (45, 84, 81) Prediction        29    2032.043279
                                                              before_expertized 29    3313.831728
                                                              expertized        29    3296.533489

[603390 rows x 1 columns]

In [4]:
models = df_grouped.index.get_level_values('model').unique()
splits = df_grouped.index.get_level_values('split').unique()
boxes = df_grouped.index.get_level_values('box').unique()
metrics = df_grouped.index.get_level_values('metric').unique()

print("Models disponibles :", models)
print("Splits disponibles :", splits)
print("Boxes disponibles  :", boxes)
print("Metrics disponibles:", metrics)

Models disponibles : Index(['Unet_bce_063', 'Unet_bce_080', 'Unet_bce_100', 'Unet_bce_120',
       'Unet_bce_140', 'Unet_bce_160', 'Unet_bce_180', 'Unet_bce_200',
       'Unet_bce_220', 'Unet_bce_240', 'Unet_bce_260', 'Unet_dice_060',
       'Unet_dice_080', 'Unet_dice_100', 'Unet_dice_120', 'Unet_dice_140',
       'Unet_dice_160', 'Unet_dice_180', 'Unet_dice_200', 'Unet_dice_240',
       'Unet_dice_260'],
      dtype='object', name='model')
Splits disponibles : Index(['Test', 'Val'], dtype='object', name='split')
Boxes disponibles  : Index(['230629PN018', '230629PN006', '230629PN010', '230629PN024',
       '230629PN027', '230629PN012', '230629PN014', '230629PN019',
       '230629PN031', '230629PN008'],
      dtype='object', name='box')
Metrics disponibles: Index(['Convex_Area_Hull', 'Intercept_curve_Area', 'LateralRootLength',
       'NumberOfLateralRoots', 'NumberOfOrgans', 'PrimaryRootLength',
       'RootDensity', 'TotalRootLength'],
      dtype='object', name='metric')


In [5]:
def get_root_ids(model, split, box, metric):
    return (
        df_grouped
        .xs((model, split, box, metric), level=('model', 'split', 'box', 'metric'))
        .index.get_level_values('root_ids')
        .unique()
    )


def get_sources(model, split, box, metric, root):
    return (
        df_grouped
        .xs((model, split, box, metric, root), level=('model', 'split', 'box', 'metric', 'root_ids'))
        .index.get_level_values('source')
        .unique()
    )


def plot_root(model_idx, split_idx, box_idx, metric_idx, root_idx):
    model = models[model_idx]
    split = splits[split_idx]
    box = boxes[box_idx + split_idx * 5]
    metric = metrics[metric_idx]
    roots = get_root_ids(model, split, box, metric)
    root = roots[root_idx]

    df_tmp = (
        df_grouped
        .xs((model, split, box, metric, root),
            level=('model', 'split', 'box', 'metric', 'root_ids'))
        .reset_index()
    )
    #print(df_tmp.head())

    # Map styles + offset pour le jitter en abscisse
    style_map = {
        'before_expertized': dict(color='orange', linestyle='--', marker=None,
                                  linewidth=2.5, markersize=8, alpha=0.8, xoff=-0.1),
        'expertized': dict(color='green', linestyle='-', marker='o',
                           linewidth=2.5, markersize=8, alpha=0.8, xoff=0.0),
        'Prediction': dict(color='red', linestyle='-', marker='x',
                           linewidth=2.5, markersize=8, alpha=0.8, xoff=0.1),
    }

    plt.figure(figsize=(16, 8))
    for source, style in style_map.items():
        df_s = df_tmp[df_tmp['source'] == source]
        if df_s.empty:
            continue

        # convertir time en float si nécessaire
        x = pd.to_numeric(df_s['time'], errors='coerce').astype(float)
        # appliquer le jitter
        x = x + style['xoff']

        plt.plot(
            x, df_s['value'],
            label=source,
            color=style['color'],
            linestyle=style['linestyle'],
            marker=style['marker'],
            linewidth=style['linewidth'],
            markersize=style['markersize'],
            alpha=style['alpha']
        )

        # annoter le dernier point
        last_x = x.iloc[-1]
        last_y = df_s['value'].iloc[-1]
        plt.text(
            last_x, last_y,
            f" {source}",
            va='center',
            fontsize=9,
            color=style['color']
        )

    # Remettre les vraies valeurs de time en ticks
    uniq_times = sorted(df_tmp['time'].unique())
    plt.xticks(uniq_times, uniq_times)

    plt.title(f"{metric} – {model} / {split} / {box} / root {root}")
    plt.xlabel('Temps')
    plt.ylabel('Value')
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.legend(title='Source')
    plt.tight_layout()
    plt.show()

In [ ]:
model_slider = widgets.IntSlider(min=0, max=len(models) - 1, step=1, value=0, description='Model')
split_slider = widgets.IntSlider(min=0, max=len(splits) - 1, step=1, value=0, description='Split')
box_slider = widgets.IntSlider(min=0, max=4, step=1, value=0, description='Box')
metric_slider = widgets.IntSlider(min=0, max=len(metrics) - 1, step=1, value=0, description='Metric')

# Slider “root” : on fixe temporairement sa plage max à 0…9 – on mettra à jour dynamiquement
root_slider = widgets.IntSlider(min=0, max=0, step=1, value=0, description='Root')


# Callback pour mettre à jour le slider root_ids quand on change l’une des 4 premières valeurs
def update_root_slider(*args):
    roots = get_root_ids(
        models[model_slider.value],
        splits[split_slider.value],
        boxes[box_slider.value + split_slider.value * 5],
        metrics[metric_slider.value]
    )
    root_slider.max = len(roots) - 1
    if (root_slider.value >= root_slider.max):
        root_slider.value = root_slider.max


# On observe les changements
for w in (model_slider, split_slider, box_slider, metric_slider):
    w.observe(update_root_slider, 'value')

# Lancer la première initialisation
update_root_slider()

# 6) Affichage interactif
ui = widgets.HBox([model_slider, split_slider, box_slider, metric_slider, root_slider])
out = widgets.interactive_output(
    plot_root,
    {
        'model_idx': model_slider,
        'split_idx': split_slider,
        'box_idx': box_slider,
        'metric_idx': metric_slider,
        'root_idx': root_slider
    }
)

display(ui, out)

Output()

## Machin

In [7]:
df_plant_wid

,model,split,box,metric,time,root_ids,source,value
0,Unet_bce_180,Val,230629PN031,NumberOfOrgans,1,"(86, 84, 81)",Prediction,1.000000
1,Unet_bce_180,Val,230629PN031,NumberOfOrgans,1,"(86, 84, 81)",expertized,1.000000
2,Unet_bce_180,Val,230629PN031,NumberOfOrgans,1,"(86, 84, 81)",before_expertized,1.000000
3,Unet_bce_180,Val,230629PN031,NumberOfOrgans,1,"(48, 47, 45)",Prediction,1.000000
4,Unet_bce_180,Val,230629PN031,NumberOfOrgans,1,"(48, 47, 45)",expertized,1.000000
...,...,...,...,...,...,...,...,...
603385,Unet_dice_120,Test,230629PN018,LateralRootLength,29,"(4, 7, 4)",expertized,102.080352
603386,Unet_dice_120,Test,230629PN018,LateralRootLength,29,"(4, 7, 4)",before_expertized,16.498474
603387,Unet_dice_120,Test,230629PN018,LateralRootLength,29,"(1, 1, 1)",Prediction,170.444195
603388,Unet_dice_120,Test,230629PN018,LateralRootLength,29,"(1, 1, 1)",expertized,317.155933


## normalized data frame

In [8]:
df_normalized = df_plant_wid.copy()
# Get the max value for each metric (regardless of time, model, or source)
max_value_for_each_metric = (
    df_normalized.groupby('metric')['value']
    .max()
    .reset_index()
    .rename(columns={'value': 'max_value'})
)

# Merge to assign max_value to each row
df_normalized = df_normalized.merge(max_value_for_each_metric, on='metric', how='left')
df_normalized['value'] = df_normalized['value'] / df_normalized['max_value']
df_normalized = df_normalized.drop(columns=['max_value'])
df_normalized

,model,split,box,metric,time,root_ids,source,value
0,Unet_bce_180,Val,230629PN031,NumberOfOrgans,1,"(86, 84, 81)",Prediction,0.034483
1,Unet_bce_180,Val,230629PN031,NumberOfOrgans,1,"(86, 84, 81)",expertized,0.034483
2,Unet_bce_180,Val,230629PN031,NumberOfOrgans,1,"(86, 84, 81)",before_expertized,0.034483
3,Unet_bce_180,Val,230629PN031,NumberOfOrgans,1,"(48, 47, 45)",Prediction,0.034483
4,Unet_bce_180,Val,230629PN031,NumberOfOrgans,1,"(48, 47, 45)",expertized,0.034483
...,...,...,...,...,...,...,...,...
603385,Unet_dice_120,Test,230629PN018,LateralRootLength,29,"(4, 7, 4)",expertized,0.030296
603386,Unet_dice_120,Test,230629PN018,LateralRootLength,29,"(4, 7, 4)",before_expertized,0.004897
603387,Unet_dice_120,Test,230629PN018,LateralRootLength,29,"(1, 1, 1)",Prediction,0.050586
603388,Unet_dice_120,Test,230629PN018,LateralRootLength,29,"(1, 1, 1)",expertized,0.094128


## Mean (not normalized) dataframe (mean for all plants)

In [9]:
# %% Agrégation par temps / model / metric
df_mean = (
    df_grouped
    .reset_index()  # remettre les colonnes
    .groupby(['model', 'metric', 'time', 'source'], as_index=False)
    .agg(mean_value=('value', 'mean'),
         std_value=('value', 'std')
         )
)

df_mean

,model,metric,time,source,mean_value,std_value
0,Unet_bce_063,Convex_Area_Hull,1,Prediction,248.365068,473.564236
1,Unet_bce_063,Convex_Area_Hull,1,before_expertized,373.695785,35.525987
2,Unet_bce_063,Convex_Area_Hull,1,expertized,373.695785,35.525987
3,Unet_bce_063,Convex_Area_Hull,2,Prediction,370.011004,833.547424
4,Unet_bce_063,Convex_Area_Hull,2,before_expertized,427.048242,32.648650
...,...,...,...,...,...,...
14611,Unet_dice_260,TotalRootLength,28,before_expertized,1651.908930,916.730044
14612,Unet_dice_260,TotalRootLength,28,expertized,1817.790345,916.273673
14613,Unet_dice_260,TotalRootLength,29,Prediction,898.506573,483.913879
14614,Unet_dice_260,TotalRootLength,29,before_expertized,1737.399024,960.907567


In [ ]:
all_models = sorted(df_mean['model'].unique().tolist())

# styles fixes par source (identiques pour tous les modèles)
source_styles = {
    'Prediction': dict(linestyle='-', marker='o'),
}
source_order = ['Prediction']


def plot_mean_all_models(metric_idx):
    metric = metrics[metric_idx]  # slider sur la métrique uniquement
    df_plot = df_mean[df_mean['metric'] == metric].copy()
    if df_plot.empty:
        print(f"Aucune donnée pour la métrique {metric}.")
        return

    # s'assurer que time est numérique et trié
    df_plot['time'] = pd.to_numeric(df_plot['time'], errors='coerce')
    df_plot = df_plot.sort_values(['model', 'source', 'time'])

    plt.figure(figsize=(14, 7))

    # tracer chaque modèle avec une couleur différente,
    # et pour chaque source un style différent
    for m in all_models:
        d_m = df_plot[df_plot['model'] == m]
        if d_m.empty:
            continue
        for src in source_order:
            d = d_m[d_m['source'] == src]
            if d.empty:
                continue
            st = source_styles.get(src, dict(linestyle='-', marker=None))
            # on utilise le cycle de couleurs de Matplotlib: une couleur par modèle
            plt.plot(
                d['time'], d['mean_value'],
                label=f"{m} – {src}",
                linewidth=2.0, markersize=6, alpha=0.95,
                **st
            )

    plt.title(f"Évolution de la valeur moyenne – métrique: {metric}")
    plt.xlabel('Time')
    plt.ylabel('Mean value')
    plt.grid(True, linestyle=':', alpha=0.5)

    # Légende compacte
    plt.legend(title='Model – Source', fontsize=9, ncol=2)
    plt.tight_layout()
    plt.show()


# widget: slider uniquement sur la métrique
widgets.interactive(
    plot_mean_all_models,
    metric_idx=metric_slider
)


interactive(children=(IntSlider(value=0, description='Metric', max=7), Output()), _dom_classes=('widget-intera…

In [ ]:
all_models = sorted(df_mean['model'].unique().tolist())

# styles fixes par source (identiques pour tous les modèles)
source_styles = {
    'Prediction': dict(linestyle='-', marker='o'),
}
source_order = ['Prediction']
colormap = plt.get_cmap("tab10")


def plot_mean_all_models(metric_idx):
    metric = metrics[metric_idx]  # slider sur la métrique uniquement
    df_plot = df_mean[df_mean['metric'] == metric].copy()
    if df_plot.empty:
        print(f"Aucune donnée pour la métrique {metric}.")
        return

    # s'assurer que time est numérique et trié
    df_plot['time'] = pd.to_numeric(df_plot['time'], errors='coerce')
    df_plot = df_plot.sort_values(['model', 'source', 'time'])

    plt.figure(figsize=(14, 7))

    # tracer chaque modèle avec une couleur différente,
    # et pour chaque source un style différent
    for m, style in zip(all_models, colormap.colors):
        d_m = df_plot[df_plot['model'] == m]
        if d_m.empty:
            continue
        for src in source_order:
            d = d_m[d_m['source'] == src]
            if d.empty:
                continue
            st = source_styles.get(src, dict(linestyle='-', marker=None))
            # on utilise le cycle de couleurs de Matplotlib: une couleur par modèle
            plt.plot(
                d['time'], d['mean_value'],
                label=f"{m}",
                linestyle=st['linestyle'], marker=st['marker'],
                color=style
            )
            # std
            plt.fill_between(
                d['time'], d['mean_value'] - d['std_value'],
                           d['mean_value'] + d['std_value'],
                alpha=0.2, color=style
            )

    plt.title(f"Évolution de la valeur moyenne – métrique: {metric}")
    plt.xlabel('Time')
    plt.ylabel('Mean value')
    plt.grid(True, linestyle=':', alpha=0.5)

    # Légende compacte
    plt.legend(title='Model – Source', fontsize=9, ncol=2)
    plt.tight_layout()
    plt.show()


# widget: slider uniquement sur la métrique
widgets.interactive(
    plot_mean_all_models,
    metric_idx=metric_slider
)

interactive(children=(IntSlider(value=0, description='Metric', max=7), Output()), _dom_classes=('widget-intera…

In [ ]:
# for each model and metric, plot the evolution over time of mean value

def plot_mean_evolution(model_idx, metric_idx):
    model = models[model_idx]
    metric = metrics[metric_idx]
    df_plot = df_mean[(df_mean['model'] == model) & (df_mean['metric'] == metric)]
    plt.figure(figsize=(16, 10))
    for source, style in zip(['Prediction', 'before_expertized', 'expertized'],
                             [{'color': 'red', 'linestyle': '-', 'marker': 'x'},
                              {'color': 'orange', 'linestyle': '--', 'marker': None},
                              {'color': 'green', 'linestyle': '-', 'marker': 'o'}]):
        df_s = df_plot[df_plot['source'] == source]
        if df_s.empty:
            continue
        plt.errorbar(
            df_s['time'], df_s['mean_value'],
            label=source,
            color=style['color'],
            linestyle=style['linestyle'],
            marker=style['marker'],
            linewidth=2.5,
            markersize=8,
            alpha=0.8
        )
        plt.fill_between(
            df_s['time'], df_s['mean_value'] - df_s['std_value'],
                          df_s['mean_value'] + df_s['std_value'],
            alpha=0.2, color=style['color']
        )
    plt.title(f"Mean evolution over time – {model} / {metric}")
    plt.xlabel('Time')
    plt.ylabel('Mean Value')
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.legend(title='Source')
    plt.tight_layout()
    plt.show()


widgets.interactive(
    plot_mean_evolution,
    model_idx=model_slider,
    metric_idx=metric_slider
)



interactive(children=(IntSlider(value=0, description='Model', max=20), IntSlider(value=0, description='Metric'…

## Error

In [13]:
# Pivot the dataframe to get 'Prediction', 'expertized', and 'real' as columns
df_pivot = (
    df_plant_wid
    .pivot_table(
        index=['model', 'metric', 'time', 'split', 'box', 'root_ids'],
        columns='source',
        values='value'
    )
    .reset_index()
)

# Compute errors only if both columns exist
df_pivot['abs_error'] = (df_pivot['expertized'] - df_pivot['Prediction']).abs()
df_pivot['real_error'] = df_pivot['expertized'] - df_pivot['Prediction']

df_error = df_pivot[
    ['model', 'metric', 'time', 'split', 'box', 'root_ids', 'abs_error', 'real_error']
]
df_error

source,model,metric,time,split,box,root_ids,abs_error,real_error
0,Unet_bce_063,Convex_Area_Hull,1,Test,230629PN018,"(12, 1, 1)",384.042295,384.042295
1,Unet_bce_063,Convex_Area_Hull,1,Test,230629PN018,"(14, 23, 13)",249.387008,249.387008
2,Unet_bce_063,Convex_Area_Hull,1,Test,230629PN018,"(16, 7, 4)",283.571320,283.571320
3,Unet_bce_063,Convex_Area_Hull,1,Test,230629PN018,"(18, 14, 8)",287.543219,287.543219
4,Unet_bce_063,Convex_Area_Hull,1,Val,230629PN006,"(1, 31, 15)",1137.069346,-1137.069346
...,...,...,...,...,...,...,...,...
201125,Unet_dice_260,TotalRootLength,29,Val,230629PN031,"(1, 1, 1)",1185.991725,1185.991725
201126,Unet_dice_260,TotalRootLength,29,Val,230629PN031,"(11, 19, 18)",3093.139972,3093.139972
201127,Unet_dice_260,TotalRootLength,29,Val,230629PN031,"(22, 47, 45)",1406.824655,1406.824655
201128,Unet_dice_260,TotalRootLength,29,Val,230629PN031,"(34, 66, 63)",1002.960697,1002.960697


In [14]:
df_mean_error = (
    df_error
    .groupby(['model', 'metric', 'time'], as_index=False)
    .agg(
        abs_error_mean=('abs_error', 'mean'),
        real_error_mean=('real_error', 'mean'),
        std_abs_error=('abs_error', 'std'),
    )
)
df_mean_error

,model,metric,time,abs_error_mean,real_error_mean,std_abs_error
0,Unet_bce_063,Convex_Area_Hull,1,378.012794,125.330717,288.143974
1,Unet_bce_063,Convex_Area_Hull,2,535.444410,57.037238,607.904584
2,Unet_bce_063,Convex_Area_Hull,3,590.100113,111.265834,595.246824
3,Unet_bce_063,Convex_Area_Hull,4,617.120617,138.354077,585.430176
4,Unet_bce_063,Convex_Area_Hull,5,636.068931,157.807904,578.561516
...,...,...,...,...,...,...
4867,Unet_dice_260,TotalRootLength,25,663.602776,663.602776,412.527491
4868,Unet_dice_260,TotalRootLength,26,745.788000,745.788000,457.664429
4869,Unet_dice_260,TotalRootLength,27,842.619166,842.619166,517.907671
4870,Unet_dice_260,TotalRootLength,28,944.038943,944.038943,594.346348


In [ ]:
all_models = sorted(df_mean_error['model'].unique().tolist())


def plot_distance_all_models(metric_idx):
    metric = metrics[metric_idx]
    df_plot = df_mean_error[df_mean_error['metric'] == metric].copy()
    if df_plot.empty:
        print(f"Aucune donnée pour la métrique {metric}.")
        return

    df_plot['time'] = pd.to_numeric(df_plot['time'], errors='coerce')
    df_plot = df_plot.sort_values(['model', 'time'])

    plt.figure(figsize=(14, 7))
    for m in all_models:
        d = df_plot[df_plot['model'] == m]
        if d.empty:
            continue
        plt.plot(
            d['time'], d['abs_error_mean'],
            marker='o', linewidth=2.0, markersize=6, alpha=0.95, label=m
        )

        plt.fill_between(
            d['time'], d['abs_error_mean'] - d['std_abs_error'],
                       d['abs_error_mean'] + d['std_abs_error'],
            alpha=0.2
        )

    plt.title(f"Évolution de l'erreur absolue moyenne – métrique: {metric}")
    plt.xlabel('Time')
    plt.ylabel('Mean absolute error')
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.legend(title='Model', ncol=2, fontsize=9)
    plt.tight_layout()
    plt.show()


# on garde le slider métrique existant
widgets.interactive(
    plot_distance_all_models,
    metric_idx=metric_slider
)


interactive(children=(IntSlider(value=0, description='Metric', max=7), Output()), _dom_classes=('widget-intera…

## Each model mean error normalization

In [16]:
df_max_value_for_each_metric = (
    df_mean_error
    .groupby(['metric'])['abs_error_mean']
    .max()
)
df_max_value_for_each_metric

metric
Convex_Area_Hull        1281.322466
Intercept_curve_Area       3.836151
LateralRootLength        878.162937
NumberOfLateralRoots      10.720930
NumberOfOrgans            10.720930
PrimaryRootLength        549.142763
RootDensity                0.361574
TotalRootLength         1389.453691
Name: abs_error_mean, dtype: float64

In [17]:
#df_max_value_for_each_metric = (
#    df_mean_error
#    .groupby(['metric'])['abs_error_mean']
#    .max()
#)

df_max_value_for_each_metric = (
    df_mean[df_mean['source'] == 'expertized']
    .groupby('metric')['mean_value']
    .max()
)

df_max_value_for_each_metric

metric
Convex_Area_Hull        1737.476559
Intercept_curve_Area       5.613427
LateralRootLength       1111.307546
NumberOfLateralRoots      13.682927
NumberOfOrgans            14.682927
PrimaryRootLength        868.337340
RootDensity                1.076263
TotalRootLength         1979.644887
Name: mean_value, dtype: float64

In [18]:
df_normalized = df_mean_error.copy()
# Create a MultiIndex for mapping
df_normalized['max_value'] = df_normalized.set_index(['metric']).index.map(df_max_value_for_each_metric)
df_normalized['abs_error_mean'] = 1 - df_normalized['abs_error_mean'] / df_normalized['max_value']
df_normalized['real_error_mean'] = df_normalized['real_error_mean'] / df_normalized['max_value']
df_normalized['std_abs_error'] = df_normalized['std_abs_error'] / df_normalized['max_value']
df_normalized = df_normalized.drop(columns=['max_value'])

# rename abs_error_mean to normalized_abs_error_mean
df_normalized = df_normalized.rename(columns={'abs_error_mean': 'norm_abs_approx'})
df_normalized

,model,metric,time,norm_abs_approx,real_error_mean,std_abs_error
0,Unet_bce_063,Convex_Area_Hull,1,0.782436,0.072134,0.165840
1,Unet_bce_063,Convex_Area_Hull,2,0.691826,0.032828,0.349878
2,Unet_bce_063,Convex_Area_Hull,3,0.660369,0.064039,0.342593
3,Unet_bce_063,Convex_Area_Hull,4,0.644818,0.079629,0.336943
4,Unet_bce_063,Convex_Area_Hull,5,0.633912,0.090826,0.332990
...,...,...,...,...,...,...
4867,Unet_dice_260,TotalRootLength,25,0.664787,0.335213,0.208385
4868,Unet_dice_260,TotalRootLength,26,0.623272,0.376728,0.231185
4869,Unet_dice_260,TotalRootLength,27,0.574358,0.425642,0.261616
4870,Unet_dice_260,TotalRootLength,28,0.523127,0.476873,0.300229


In [19]:
# %% Optionnel : visualisation de l'évolution de la distance moyenne au cours du temps

def plot_distance_evolution(model_idx, metric_idx):
    model = models[model_idx]
    metric = metrics[metric_idx]
    df_plot = df_normalized[(df_normalized['model'] == model) & (df_normalized['metric'] == metric)]
    if df_plot.empty:
        print("Aucune paire Prediction/expertized trouvée pour cette combinaison.")
        return
    plt.figure(figsize=(14, 7))
    plt.errorbar(
        df_plot['time'], df_plot['norm_abs_approx'],
        label='|Prediction - expertized| / max_{metric, time} (|Prediction - expertized|)',
        linestyle='-', marker='o', linewidth=2.5, markersize=7, alpha=0.9
    )
    plt.fill_between(
        df_plot['time'],
        df_plot['norm_abs_approx'] - df_plot['std_abs_error'],
        df_plot['norm_abs_approx'] + df_plot['std_abs_error'],
        alpha=0.2, color='blue'
    )
    plt.title(f"Évolution de l'erreur absolue normalisée – {model} / {metric}")
    plt.xlabel('Time')
    plt.ylabel('Norm. mean absolute error')
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.legend()
    plt.ylim(0, 1.2)
    plt.tight_layout()
    plt.show()


widgets.interactive(
    plot_distance_evolution,
    model_idx=model_slider,
    metric_idx=metric_slider
)


interactive(children=(IntSlider(value=0, description='Model', max=20), IntSlider(value=0, description='Metric'…

In [ ]:
def plot_distance_all_models(metric_idx):
    metric = metrics[metric_idx]
    df_plot = df_normalized[df_normalized['metric'] == metric].copy()
    if df_plot.empty:
        print(f"Aucune donnée pour la métrique '{metric}'.")
        return

    # on s'assure que time est numérique et trié par modèle
    df_plot['time'] = pd.to_numeric(df_plot['time'], errors='coerce')
    df_plot = df_plot.sort_values(['model', 'time'])

    # couleurs auto (une par modèle)
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    models_in_plot = df_plot['model'].unique().tolist()

    plt.figure(figsize=(14, 7))
    for i, m in enumerate(models_in_plot):
        d = df_plot[df_plot['model'] == m]
        if d.empty:
            continue

        # courbe + barres d'erreur (plus lisible que des aplats multiples)
        plt.errorbar(
            d['time'], d['norm_abs_approx'],
            yerr=d.get('std_abs_error', None),
            label=m,
            linestyle='-',
            marker='o',
            linewidth=2.0,
            markersize=6,
            alpha=0.9,
            color=colors[i % len(colors)],
            capsize=3,
        )

    plt.title(f"Évolution de l'erreur absolue normalisée — métrique: {metric}")
    plt.xlabel('Time')
    plt.ylabel('Norm. mean absolute error')
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.legend(title='Model', ncol=2, fontsize=9)
    plt.ylim(0, 1)  # fixe l'ordonnée entre 0 et 1
    plt.tight_layout()
    plt.show()


# Slider uniquement pour la métrique
widgets.interactive(
    plot_distance_all_models,
    metric_idx=metric_slider
)

interactive(children=(IntSlider(value=0, description='Metric', max=7), Output()), _dom_classes=('widget-intera…

## Error at all time

In [21]:
df_norm_mean_at_all_time = df_normalized.groupby(['metric', 'model'])['norm_abs_approx'].mean()
df_norm_mean_at_all_time

metric            model        
Convex_Area_Hull  Unet_bce_063     0.496206
                  Unet_bce_080     0.813779
                  Unet_bce_100     0.987653
                  Unet_bce_120     0.988119
                  Unet_bce_140     0.988791
                                     ...   
TotalRootLength   Unet_dice_160    0.858043
                  Unet_dice_180    0.860337
                  Unet_dice_200    0.859621
                  Unet_dice_240    0.977288
                  Unet_dice_260    0.859501
Name: norm_abs_approx, Length: 168, dtype: float64

In [22]:
all_models = sorted(df_normalized['model'].unique().tolist())
print(all_models)

['Unet_bce_063', 'Unet_bce_080', 'Unet_bce_100', 'Unet_bce_120', 'Unet_bce_140', 'Unet_bce_160', 'Unet_bce_180', 'Unet_bce_200', 'Unet_bce_220', 'Unet_bce_240', 'Unet_bce_260', 'Unet_dice_060', 'Unet_dice_080', 'Unet_dice_100', 'Unet_dice_120', 'Unet_dice_140', 'Unet_dice_160', 'Unet_dice_180', 'Unet_dice_200', 'Unet_dice_240', 'Unet_dice_260']


In [23]:
import os
import pandas as pd


def extract_last_metrics(base_path):
    """
    Explore récursivement base_path pour trouver les fichiers CSV de scalars
    et construit un dictionnaire de dictionnaires : 
    {metric_name: {model_name: last_value}}
    """
    # conteneur principal : metric -> modèle -> valeur
    metrics_dict = {}

    # Parcours des répertoires de modèles
    for model_name in os.listdir(base_path):
        model_path = os.path.join(base_path, model_name, "logs", "scalars")
        if not os.path.isdir(model_path):
            continue  # on saute ce qui n'est pas un répertoire valide

        # Parcours des fichiers CSV de ce modèle
        for file in os.listdir(model_path):
            if not file.endswith(".csv"):
                continue

            file_path = os.path.join(model_path, file)
            try:
                df = pd.read_csv(file_path)
                if "value" not in df.columns or df.empty:
                    continue
                last_value = df["value"].iloc[-1]

                # Nettoyage du nom de la métrique
                metric_name = file.replace("val_", "").replace(".csv", "")

                # Initialisation si besoin
                if metric_name not in metrics_dict:
                    metrics_dict[metric_name] = {}

                metrics_dict[metric_name][model_name] = float(last_value)

            except Exception as e:
                print(f"⚠️ Impossible de lire {file_path}: {e}")

    return metrics_dict

def extract_metrics_epoch(base_path, epoch_num):
    """
    3rd column is called 'step', Explore récursivement base_path pour trouver les fichiers CSV de scalars jusqu'à step = epoch_num
    """
    # conteneur principal : metric -> modèle -> valeur
    metrics_dict = {}

    # Parcours des répertoires de modèles
    for model_name in os.listdir(base_path):
        model_path = os.path.join(base_path, model_name, "logs", "scalars")
        if not os.path.isdir(model_path):
            continue  # on saute ce qui n'est pas un répertoire valide

        # Parcours des fichiers CSV de ce modèle
        for file in os.listdir(model_path):
            if not file.endswith(".csv"):
                continue

            file_path = os.path.join(model_path, file)
            try:
                df = pd.read_csv(file_path)
                if "value" not in df.columns or df.empty:
                    continue
                if epoch_num >= len(df):
                    print(f"⚠️ Pas assez d'époques dans {file_path}")
                    return extract_last_metrics(base_path)
                try:
                    epoch_value = df["value"].iloc[df["step"].iloc[epoch_num - 2]]

                except Exception as e:
                    print(f"⚠️ Impossible de lire {file_path}: {e}")
                    extract_last_metrics(base_path)

                # Nettoyage du nom de la métrique
                metric_name = file.replace("val_", "").replace(".csv", "")

                # Initialisation si besoin
                if metric_name not in metrics_dict:
                    metrics_dict[metric_name] = {}

                metrics_dict[metric_name][model_name] = float(epoch_value)

            except Exception as e:
                print(f"⚠️ Impossible de lire {file_path}: {e}")

    return metrics_dict


def extract_metrics_corresponding_epoch(base_path, all_models): # ['Unet_bce_063', 'Unet_bce_080', ...]
    # extract epoch from each model
    all_models_epochs = [int(model.split('_')[-1]) for model in all_models]
    model_names = [model.split('_')[0] + '_' + model.split('_')[1] for model in all_models]
    metrics_dict = {}
    
    # Parcours des répertoires de modèles
    for model_name in os.listdir(base_path):
        model_path = os.path.join(base_path, model_name, "logs", "scalars")
        if not os.path.isdir(model_path):
            continue  # on saute ce qui n'est pas un répertoire valide

        # on vérifie que le nom du modèle est dans model_names
        if model_name not in model_names:
            continue
        
        for model in all_models:

            if model.startswith(model_name):
                epoch_num = all_models_epochs[all_models.index(model)]
            else:
                continue
            

            # Parcours des fichiers CSV de ce modèle
            for file in os.listdir(model_path):
                if not file.endswith(".csv"):
                    continue

                file_path = os.path.join(model_path, file)
                try:
                    df = pd.read_csv(file_path)
                    if "value" not in df.columns or df.empty:
                        continue
                    if epoch_num >= len(df):
                        print(f"⚠️ Pas assez d'époques dans {file_path}")
                        return extract_last_metrics(base_path)
                    try:
                        epoch_value = df["value"].iloc[df["step"].iloc[epoch_num - 2]]

                    except Exception as e:
                        print(f"⚠️ Impossible de lire {file_path}: {e}")
                        extract_last_metrics(base_path)

                    # Nettoyage du nom de la métrique
                    metric_name = file.replace("val_", "").replace(".csv", "")

                    # Initialisation si besoin
                    if metric_name not in metrics_dict:
                        metrics_dict[metric_name] = {}

                    metrics_dict[metric_name][all_models[all_models.index(model)]] = float(epoch_value)

                except Exception as e:
                    print(f"⚠️ Impossible de lire {file_path}: {e}")

    return metrics_dict

base_path = os.path.expanduser("~/Documents/code/RSMLExtraction/Results/Training/Logs/")
cv_metrics_results = extract_metrics_corresponding_epoch( base_path, all_models=all_models)
print(cv_metrics_results)


{'Specificity': {'Unet_dice_060': 0.999332428, 'Unet_dice_080': 0.9990941286, 'Unet_dice_100': 0.9992593527, 'Unet_dice_120': 0.9992260933, 'Unet_dice_140': 0.9992359877, 'Unet_dice_160': 0.9992204905, 'Unet_dice_180': 0.9992747307, 'Unet_dice_200': 0.999235034, 'Unet_dice_240': 0.9992594719, 'Unet_dice_260': 0.9992449284, 'Unet_bce_063': 0.9982545972, 'Unet_bce_080': 0.9990547895, 'Unet_bce_100': 0.9991066456, 'Unet_bce_120': 0.9992212653, 'Unet_bce_140': 0.999260664, 'Unet_bce_160': 0.9992787838, 'Unet_bce_180': 0.9992935658, 'Unet_bce_200': 0.9992684126, 'Unet_bce_220': 0.9992833138, 'Unet_bce_240': 0.9992842674, 'Unet_bce_260': 0.9992832541}, 'HausdorffDistance': {'Unet_dice_060': 33.92109299, 'Unet_dice_080': 32.39938354, 'Unet_dice_100': 30.85949326, 'Unet_dice_120': 30.33571625, 'Unet_dice_140': 30.86873245, 'Unet_dice_160': 31.0152626, 'Unet_dice_180': 30.62447166, 'Unet_dice_200': 30.64321136, 'Unet_dice_240': 30.75297356, 'Unet_dice_260': 30.73775291, 'Unet_bce_063': 43.88248

In [24]:
df_cv_and_graph = (
    df_normalized
    .groupby(['metric', 'model'])[['norm_abs_approx', 'std_abs_error']]
    .mean()
    .reset_index()
)

#for metric, values_and_model in zip(metric_name, all_models_to_X):
#    df_cv_and_graph[metric] = df_cv_and_graph['model'].map(values_and_model)
for metric, model_values in cv_metrics_results.items():
    df_cv_and_graph[metric] = df_cv_and_graph['model'].map(model_values)
df_cv_and_graph

,metric,model,norm_abs_approx,std_abs_error,Specificity,HausdorffDistance,PersistenceWassersteinGPUParallel_0,Betti1JaccardRatioGPU,PixelAccuracy,AverageCenterlineDistance,...,MeanIoU,PersistenceBottleneckGPUParallel_0,Precision,Dice,train_batch_loss,Betti0VariationIndexGPU,Betti0RelativeErrorGPU,loss_DiceLoss,Betti1RelativeErrorGPU,loss_BCE
0,Convex_Area_Hull,Unet_bce_063,0.496206,0.285148,0.998255,43.882484,3.113853,0.540795,0.998109,0.881357,...,0.841256,0.0,0.870013,0.912864,0.018722,0.114021,0.213660,NaN,15625000.00,0.189116
1,Convex_Area_Hull,Unet_bce_080,0.813779,0.234514,0.999055,40.538422,1.933344,0.527277,0.998515,0.725073,...,0.873941,0.0,0.923116,0.932233,0.018692,0.105613,0.222402,NaN,13671875.00,0.148462
2,Convex_Area_Hull,Unet_bce_100,0.987653,0.034750,0.999107,37.908966,1.749899,0.563080,0.998595,0.704193,...,0.877823,0.0,0.927086,0.934277,0.014137,0.104621,0.242148,NaN,8800552.00,0.140507
3,Convex_Area_Hull,Unet_bce_120,0.988119,0.034824,0.999221,29.590921,1.995386,0.588732,0.998725,0.549670,...,0.889210,0.0,0.936164,0.940857,0.015366,0.119828,0.238163,NaN,8593750.00,0.127485
4,Convex_Area_Hull,Unet_bce_140,0.988791,0.034881,0.999261,29.647942,1.680564,0.594844,0.998733,0.546737,...,0.890054,0.0,0.939025,0.941331,0.013568,0.109958,0.211997,NaN,4296875.00,0.126741
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163,TotalRootLength,Unet_dice_160,0.858043,0.098415,0.999220,31.015263,1.770426,0.570069,0.998730,0.553701,...,0.889631,0.0,0.936074,0.941100,0.660066,0.108600,0.236427,0.058900,2343750.25,NaN
164,TotalRootLength,Unet_dice_180,0.860337,0.095864,0.999275,30.624472,1.649088,0.571053,0.998720,0.552295,...,0.889205,0.0,0.939985,0.940883,0.624332,0.103119,0.224317,0.059117,2734375.25,NaN
165,TotalRootLength,Unet_dice_200,0.859621,0.095390,0.999235,30.643211,1.675724,0.588185,0.998730,0.551786,...,0.889722,0.0,0.937128,0.941157,0.597925,0.104791,0.231425,0.058843,2343750.00,NaN
166,TotalRootLength,Unet_dice_240,0.977288,0.025311,0.999259,30.752974,1.639791,0.586675,0.998724,0.552847,...,0.889396,0.0,0.938878,0.940984,0.559034,0.103355,0.228039,0.059016,2343750.00,NaN


In [25]:
df_cv_and_graph_normalize_for_all_cv_metrics = df_cv_and_graph.copy()

for metric in cv_metrics_results.keys():
    if (df_cv_and_graph_normalize_for_all_cv_metrics[metric] > 1).any() or (
            df_cv_and_graph_normalize_for_all_cv_metrics[metric] < 0).any():
        max_value = df_cv_and_graph_normalize_for_all_cv_metrics[metric].max()
        df_cv_and_graph_normalize_for_all_cv_metrics[metric] = 1 - (
                df_cv_and_graph_normalize_for_all_cv_metrics[metric] / max_value
        )
df_cv_and_graph_normalize_for_all_cv_metrics

,metric,model,norm_abs_approx,std_abs_error,Specificity,HausdorffDistance,PersistenceWassersteinGPUParallel_0,Betti1JaccardRatioGPU,PixelAccuracy,AverageCenterlineDistance,...,MeanIoU,PersistenceBottleneckGPUParallel_0,Precision,Dice,train_batch_loss,Betti0VariationIndexGPU,Betti0RelativeErrorGPU,loss_DiceLoss,Betti1RelativeErrorGPU,loss_BCE
0,Convex_Area_Hull,Unet_bce_063,0.496206,0.285148,0.998255,0.000000,0.000000,0.540795,0.998109,0.881357,...,0.841256,0.0,0.870013,0.912864,0.018722,0.114021,0.213660,NaN,0.000000,0.189116
1,Convex_Area_Hull,Unet_bce_080,0.813779,0.234514,0.999055,0.076205,0.379115,0.527277,0.998515,0.725073,...,0.873941,0.0,0.923116,0.932233,0.018692,0.105613,0.222402,NaN,0.125000,0.148462
2,Convex_Area_Hull,Unet_bce_100,0.987653,0.034750,0.999107,0.136125,0.438028,0.563080,0.998595,0.704193,...,0.877823,0.0,0.927086,0.934277,0.014137,0.104621,0.242148,NaN,0.436765,0.140507
3,Convex_Area_Hull,Unet_bce_120,0.988119,0.034824,0.999221,0.325678,0.359191,0.588732,0.998725,0.549670,...,0.889210,0.0,0.936164,0.940857,0.015366,0.119828,0.238163,NaN,0.450000,0.127485
4,Convex_Area_Hull,Unet_bce_140,0.988791,0.034881,0.999261,0.324379,0.460294,0.594844,0.998733,0.546737,...,0.890054,0.0,0.939025,0.941331,0.013568,0.109958,0.211997,NaN,0.725000,0.126741
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163,TotalRootLength,Unet_dice_160,0.858043,0.098415,0.999220,0.293220,0.431436,0.570069,0.998730,0.553701,...,0.889631,0.0,0.936074,0.941100,0.660066,0.108600,0.236427,0.058900,0.850000,NaN
164,TotalRootLength,Unet_dice_180,0.860337,0.095864,0.999275,0.302125,0.470403,0.571053,0.998720,0.552295,...,0.889205,0.0,0.939985,0.940883,0.624332,0.103119,0.224317,0.059117,0.825000,NaN
165,TotalRootLength,Unet_dice_200,0.859621,0.095390,0.999235,0.301698,0.461849,0.588185,0.998730,0.551786,...,0.889722,0.0,0.937128,0.941157,0.597925,0.104791,0.231425,0.058843,0.850000,NaN
166,TotalRootLength,Unet_dice_240,0.977288,0.025311,0.999259,0.299197,0.473388,0.586675,0.998724,0.552847,...,0.889396,0.0,0.938878,0.940984,0.559034,0.103355,0.228039,0.059016,0.850000,NaN


In [26]:
df_scatter = df_cv_and_graph_normalize_for_all_cv_metrics.copy()

cv_metrics = list(cv_metrics_results.keys())
graph_metrics = df_scatter["metric"].unique().tolist()

models_order = df_scatter['model'].unique().tolist()
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
color_map = {m: colors[i % len(colors)] for i, m in enumerate(models_order)}


def plot_scatter_norm_vs_external(cv_key, graph_key, show_error=True, alpha_pts=0.8, size_pts=40):
    if cv_key not in cv_metrics:
        print(f"Métrique '{cv_key}' inconnue. Choisis parmi: {cv_metrics}")
        return
    if graph_key not in graph_metrics:
        print(f"Métrique '{graph_key}' inconnue. Choisis parmi: {graph_metrics}")
        return
    d = df_scatter.dropna(subset=[cv_key, 'metric', 'norm_abs_approx']).copy()
    d = d[d['metric'] == graph_key]

    if d.empty:
        print("Nothing to see here")
        return

    plt.figure(figsize=(12, 7))
    for m in models_order:
        dm = d[d['model'] == m]
        if dm.empty:
            continue
        x = dm[cv_key].values
        y = dm['norm_abs_approx'].values
        c = color_map[m]

        plt.scatter(x, y, s=size_pts, alpha=alpha_pts, label=m, color=c)

        if show_error and 'std_abs_error' in dm.columns:
            yerr = dm['std_abs_error'].values
            plt.errorbar(x, y, yerr=yerr, fmt='none', ecolor=c, alpha=0.5, capsize=2)

    plt.title(f"norm_abs_approx {graph_key} vs {cv_key} — tous modèles")
    plt.xlabel(cv_key)
    plt.ylabel("norm_abs_approx")
    plt.grid(True, linestyle=":", alpha=0.5)
    plt.xlim(0, 1)
    plt.ylim(0, 1.1)
    plt.legend(title="Modèle", ncol=2, fontsize=9)
    plt.tight_layout()
    plt.show()


widgets.interactive(
    plot_scatter_norm_vs_external,
    cv_key=widgets.Dropdown(options=cv_metrics, value=cv_metrics[0], description="CV metric"),
    graph_key=widgets.Dropdown(options=graph_metrics, value="NumberOfOrgans", description="Graph error"),
    show_error=widgets.Checkbox(value=True, description="Barres d’erreur"),
    alpha_pts=widgets.FloatSlider(value=1.0, min=0.2, max=1.0, step=0.1, description="Alpha"),
    size_pts=widgets.IntSlider(value=40, min=10, max=120, step=5, description="Taille points"),
)

interactive(children=(Dropdown(description='CV metric', options=('Specificity', 'HausdorffDistance', 'Persiste…

In [ ]:
df_radar = df_cv_and_graph_normalize_for_all_cv_metrics.copy()

# Axes
graph_metrics_all = sorted(df_radar["metric"].dropna().unique().tolist())
cv_metrics_all = sorted(cv_metrics_results.keys())

# Modèles + couleurs stables
models_all = df_radar['model'].dropna().unique().tolist()
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
color_map = {m: colors[i % len(colors)] for i, m in enumerate(models_all)}


# --- Helpers radar ---
def _radar_angles(n):
    ang = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
    return ang + ang[:1]


def _plot_poly(ax, angles, values, label, color, fill=False, alpha=0.18, marker=None):
    vals = list(values) + [values[0]]
    ax.plot(angles, vals, linewidth=2, color=color, label=label, marker=marker)
    if fill:
        ax.fill(angles, vals, alpha=alpha, color=color)


def _setup_ax(ax, labels, title):
    angles = _radar_angles(len(labels))
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(["0.25", "0.5", "0.75", "1.0"], fontsize=8)
    ax.set_ylim(0, 1.0)
    ax.set_title(title, pad=12, fontsize=12)
    return angles


# --- Fonction de tracé ---
def plot_double_radar(models_to_show, graph_axes, cv_axes, fill_polys=True, show_legend=True):
    # Garde un ordre stable
    models = [m for m in models_all if m in set(models_to_show)]
    if len(models) == 0:
        print("Sélectionne au moins un modèle.")
        return

    graph_axes = [m for m in graph_metrics_all if m in set(graph_axes)]
    cv_axes = [m for m in cv_metrics_all if m in set(cv_axes)]
    if len(graph_axes) < 3 or len(cv_axes) < 3:
        print("Il faut au moins 3 axes par radar.")
        return

    # Prépare les valeurs par modèle
    # Radar gauche: pour chaque metric interne -> norm_abs_approx moyen (déjà en 0..1, plus haut = mieux)
    left_vals_by_model = {}
    for m in models:
        d = df_radar[df_radar['model'] == m][['metric', 'norm_abs_approx']]
        mp = dict(zip(d['metric'], d['norm_abs_approx']))
        left_vals_by_model[m] = [float(mp.get(ax, np.nan)) for ax in graph_axes]

    # Radar droit: métriques CV -> colonnes (scores 0..1, les 'error' déjà inversées plus haut)
    right_vals_by_model = {}
    for m in models:
        d = df_radar[df_radar['model'] == m].iloc[:1]  # valeurs CV identiques sur les lignes du modèle
        if d.empty:
            right_vals_by_model[m] = [np.nan] * len(cv_axes)
        else:
            row = d.iloc[0]
            right_vals_by_model[m] = [float(row[ax]) for ax in cv_axes]

    # Figure
    _ = plt.figure(figsize=(18, 10))
    ax_left = plt.subplot(1, 2, 1, projection='polar')
    ax_right = plt.subplot(1, 2, 2, projection='polar')

    # Axes / titres
    ang_left = _setup_ax(ax_left, graph_axes, "Graph metrics — norm_abs")
    ang_right = _setup_ax(ax_right, cv_axes, "CV metrics — score")

    # Un polygone par modèle sur chaque radar
    markers = ['o', 's', 'D', '^', 'v', 'X', 'P', '*']
    for i, m in enumerate(models):
        col = color_map[m]
        mrk = markers[i % len(markers)]
        # gauche
        lv = left_vals_by_model[m]
        _plot_poly(ax_left, ang_left, lv, label=m, color=col, fill=fill_polys, marker=mrk)
        # droite
        rv = right_vals_by_model[m]
        _plot_poly(ax_right, ang_right, rv, label=m, color=col, fill=fill_polys, marker=mrk)

    # Légende commune
    if show_legend:
        handles = [plt.Line2D([0], [0], marker=markers[i % len(markers)], color='w',
                              markerfacecolor=color_map[m], markersize=8, label=m)
                   for i, m in enumerate(models)]
        ax_right.legend(handles=handles, title="Modèles",
                        bbox_to_anchor=(1.25, 1.0), loc='upper left', fontsize=9)

    plt.tight_layout()
    plt.show()


# --- Widgets ---
w_models = widgets.SelectMultiple(options=models_all, value=tuple(models_all), description="Modèles",
                                  rows=min(8, len(models_all)))
w_graph = widgets.SelectMultiple(options=graph_metrics_all, value=tuple(graph_metrics_all), description="Graph axes",
                                 rows=min(10, len(graph_metrics_all)))
w_cv = widgets.SelectMultiple(options=cv_metrics_all, value=tuple(cv_metrics_all), description="CV axes",
                              rows=len(cv_metrics_all))
w_fill = widgets.Checkbox(value=True, description="Remplir")
w_legend = widgets.Checkbox(value=True, description="Légende")

widgets.interactive(
    plot_double_radar,
    models_to_show=w_models,
    graph_axes=w_graph,
    cv_axes=w_cv,
    fill_polys=w_fill,
    show_legend=w_legend
)

interactive(children=(SelectMultiple(description='Modèles', index=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 1…